This notebook helps you analyse basin at different stream orders, or create model domains

In [ ]:
# Call all libraries to be used
import pyflwdir as flwdir
import rasterio
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd

In [ ]:
# local convenience methods (see utils.py script in notebooks folder)
from utils import vectorize  # convenience method to vectorize rasters
from utils import quickplot, colors, cm  # data specific quick plot method

In [ ]:
# define current projection (input), if required
oldPP = rasterio.crs.CRS.from_string(
"+proj=laea +lat_0=5 +lon_0=20 +x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
)

Set paths for all files and outputs

In [ ]:
#global_path = "C:/Users/Edisson/OneDrive - Cardiff University/PhD/WS/LandLab/HAD/WS/input_model/"
global_path = "/home/c1755103/HAD/input/"

# output paths
output_path_rst = "/home/c1755103/HAD/Datasets/postpp/raster/" # raster files
output_path_shp = "/home/c1755103/HAD/Datasets/postpp/shp/" #shapefiles

# read datasets
fdem = global_path + "HAD_DEM_utm_mm.asc"
fflowdir = global_path + "HAD_flowdir_D8.asc"
friver = global_path + "HAD_riv_length_utm_v2.asc"
fmask = global_path + "HAD_mask_utm.asc"

foutput = global_path + "HAD_stream_order.asc"

Read all datasets

In [ ]:
# Read flow direction raster (D8 encoded)
with rasterio.open(fflowdir, "r+") as src:
    fdir = src.read(1)
    src.crs = oldPP # assign projection
    profile = src.profile
fdir = np.array(fdir, dtype=np.uint8)
fdir[fdir == 255] = 0

In [ ]:
# Read stream raster
with rasterio.open(fmask) as src:
    mask = src.read(1).astype(bool)

# Read stream raster
with rasterio.open(fmask) as src:
    maskint = src.read(1).astype(int)

In [ ]:
# Initialize flow direction object
fdir = flwdir.from_array(fdir, ftype="d8",
                         mask=mask,
                        transform=src.transform,)

In [ ]:
# specify stream order
stream_order = 5

# specify paths for output files
fbassin_sto = output_path_rst + "JU_HAD_basin_stream_order_sto_" + str(stream_order) + ".asc"
fbassin_sto_out = output_path_rst + "JU_HAD_basin_outlet_stream_order_sto_" + str(stream_order) + ".asc"
fbassin_sto_shp = output_path_shp + "JU_HAD_basin_stream_order_sto_" + str(stream_order) + ".shp"
fbassin_sto_out_shp = output_path_shp + "JU_HAD_basin_outlet_stream_order_sto_" + str(stream_order) + ".shp"


# calculate subbasins with a minimum stream order and its outlets
subbas, idxs_out = fdir.subbasins_streamorder(min_sto=stream_order, mask=None)

# transfrom map and point locations to GeoDataFrames
gdf_subbas = vectorize(subbas.astype(np.int32), 0, src.transform, crs=oldPP, name="basin")
gdf_out = gpd.GeoSeries(gpd.points_from_xy(*fdir.xy(idxs_out), crs=oldPP))

# plot
gpd_plot_kwds = dict(
    column="basin", cmap=cm.Set3, edgecolor="black", alpha=0.6, linewidth=0.5
)
bas = (gdf_subbas, gpd_plot_kwds)
points = (gdf_out, dict(color="k", markersize=20))
title = "Subbasins based on a minimum stream order"

# Save raster files
# save basins
profile.update(dtype="int32")
subbas[maskint<=0] = 0
with rasterio.open(fbassin_sto, "w", **profile) as dst:
    dst.write(subbas.astype("int32"), 1)

# save outlets
mask_outlet = np.zeros_like(subbas.flatten(), dtype=int)
mask_outlet[idxs_out] = idxs_out
mask_outlet = mask_outlet.reshape(np.shape(subbas))
maskint[maskint<=0] = 0
mask_outlet = mask_outlet*maskint
with rasterio.open(fbassin_sto_out, "w", **profile) as dst:
    dst.write(mask_outlet.astype("int32"), 1)

# save shapefiles
gdf_subbas.to_file(fbassin_sto_shp, driver='ESRI Shapefile')

gdf_out.to_file(fbassin_sto_out_shp, driver='ESRI Shapefile')

In [ ]:
print(f"The number of basins for stream order {stream_order} is {len(np.unique(subbas))}")

### Plot datasets

Plot raster file

In [ ]:
plt.imshow(subbas)

Plot shapefiles, polygon

In [ ]:
gdf_subbas.plot(
    figsize=(8, 6),
    edgecolor="black",   # polygon borders
    facecolor=None,#"lightblue",  # fill color
    alpha=0.7,           # transparency
    linewidth=0.8
)